# Evaluation 2 - Four Model Comparison and Final Model Selection

## 1. Introduction

This notebook presents the comprehensive **Evaluation 2** comparative analysis, optimization review, and final model selection for the **Hotel Booking Cancellation Prediction** project.

### Project Objective
The business goal is to accurately predict whether a scheduled hotel reservation will ultimately be cancelled (`is_canceled = 1`) or honored (`is_canceled = 0`). Accurate cancellation prediction empowers hotel revenue managers to:
- Proactively manage dynamic room overbooking strategies.
- Accurately forecast net occupancy and protect RevPAR (Revenue Per Available Room).
- Allocate housekeeping, front-desk staffing, and food & beverage logistics efficiently.
- Target high-risk reservations with tailored retention incentives or advance deposit requirements.

### Four Algorithms Implemented
In accordance with course specifications, four distinct machine learning algorithms spanning linear, single-tree, bagging, and boosting paradigms were implemented:

1. **Member 1 - Logistic Regression**: A foundational regularized linear classifier that outputs calibrated cancellation probabilities.
2. **Member 2 - Decision Tree Classifier**: A non-parametric partition model capturing hierarchical non-linear feature interactions.
3. **Member 3 - Random Forest Classifier**: An ensemble bagging technique combining decorrelated decision trees to curtail prediction variance.
4. **Member 4 - Gradient Boosting Classifier**: A sequential additive boosting technique that iteratively optimizes the pseudo-residuals of weak learners.

### Untouched Test Set Guarantee
All baseline and tuned models across all four members were evaluated on **exactly the same untouched holdout test set**:
- **Total Holdout Test Observations**: $17,446$ bookings ($20.0\%$ stratified split)
- **Actual Cancellations**: $4,802$ cancellations
- **Actual Non-Cancellations**: $12,644$ honored bookings
- **Test Cancellation Proportion**: $27.5249\%$ ($\approx 27.52\%$)
- **Target Variable**: `is_canceled` (Binary: $1 = \text{Cancelled}, 0 = \text{Not Cancelled}$)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Styling configurations (using standard matplotlib for maximum reproducibility)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.dpi'] = 120

# Locate project root and verify CSV results path
REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'modeling' else Path.cwd()
RESULTS_CSV = REPO_ROOT / 'results/modeling/model_comparison.csv'

print(f"Repository Root: {REPO_ROOT}")
print(f"Results CSV Path: {RESULTS_CSV} (Exists: {RESULTS_CSV.exists()})")


## 2. Dataset and Fair Comparison Setup

A scientifically rigorous machine learning benchmark requires a strictly controlled experimental environment. To guarantee that performance variations reflect model learning capacity rather than data discrepancies, the group enforced the following experimental controls:

| Experimental Dimension | Specification & Verification | Purpose in Fair Comparison |
| :--- | :--- | :--- |
| **Raw Dataset** | 119,390 hotel booking records (`hotel_bookings.csv`) | Standardized data foundation |
| **Data Cleaning** | 31,994 duplicates removed, 168 invalid records dropped (87,228 clean rows) | Eliminated noisy artifacts and invalid observations |
| **Data Leakage Removal** | Dropped `reservation_status` and `reservation_status_date` | Prevented deterministic post-outcome information leakage |
| **Feature Engineering** | 8 leak-free domain features (`total_guests`, `total_stay`, `is_family`, etc.) | Unified predictor representation |
| **Train/Test Partition** | 80/20 Stratified Split (`random_state=42`) | Identical training ($N=69,782$) and test ($N=17,446$) samples |
| **Class Stratification** | Exactly $27.5249\%$ cancellations in both train and test partitions | Preserved empirical class distribution across splits |
| **Preprocessing Isolation** | `ColumnTransformer` fitted strictly on $X_{\text{train}}$ only | No test-set statistical leakage into scaling or encoding |
| **Target & Positive Class** | Binary target `is_canceled`, Positive Class $= 1$ (Cancelled) | Consistent evaluation perspective across all members |
| **Scoring Consistency** | ROC-AUC computed from soft probabilities `predict_proba[:, 1]` | Avoided threshold distortion from hard discrete predictions |

> **Why the Same Test Set is Mandatory:**
> If models were tested on different subsets or different random splits, variations in sample difficulty, class ratio, or seasonal anomalies would confound the results. Benchmarking on an identical, untouched holdout partition is the cornerstone of scientific evaluation.


In [ ]:
# Dataset split and partition verification summary
split_verification = pd.DataFrame({
    'Metric / Property': [
        'Total Cleaned Dataset Size',
        'Training Set Size (80%)',
        'Holdout Test Set Size (20%)',
        'Test Actual Cancellations (Class 1)',
        'Test Actual Non-Cancellations (Class 0)',
        'Test Cancellation Proportion',
        'Target Variable Name',
        'Positive Class Label',
        'Leakage Columns Excluded',
        'Random Seed (Reproducibility)'
    ],
    'Value / Verification': [
        '87,228 rows',
        '69,782 rows',
        '17,446 rows',
        '4,802 bookings',
        '12,644 bookings',
        '27.5249% (~27.52%)',
        'is_canceled',
        '1 (Cancelled)',
        'reservation_status, reservation_status_date',
        '42 (group-wide standard)'
    ]
})
print("Dataset and Holdout Partition Verification:")
split_verification


## 3. Model Overview

Each member explored a distinct algorithmic family to provide structural diversity across linear models, single decision trees, and ensemble architectures:

### 1. Member 1 - Logistic Regression
* **Core Concept**: Models the log-odds of cancellation as a linear combination of predictors squashed through the logistic sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$.
* **Characteristics**: Fast, transparent, highly interpretable via coefficient weights, and regularized with $L_1$ or $L_2$ penalties.
* **Undergraduate Summary**: A linear baseline that calculates a weighted score and converts it into a cancellation probability between 0 and 1.

### 2. Member 2 - Decision Tree Classifier
* **Core Concept**: Recursively partitions the feature space into axis-aligned rectangular regions using purity criteria (Gini impurity or Entropy).
* **Characteristics**: Non-parametric, interpretable via conditional rules, captures non-linear feature interactions without manual interaction terms, but highly susceptible to high variance and overfitting when unconstrained.
* **Undergraduate Summary**: A hierarchical flowchart of if-else questions that splits data until leaf nodes make a majority prediction.

### 3. Member 3 - Random Forest Classifier
* **Core Concept**: A bagging (Bootstrap Aggregating) ensemble that constructs a forest of hundreds of decorrelated decision trees, sampling subsets of features at each split (`max_features='sqrt'`).
* **Characteristics**: Dramatically reduces model variance without increasing bias; robust to outliers and noisy features; aggregates votes across trees.
* **Undergraduate Summary**: A 'wisdom of the crowd' algorithm where hundreds of diverse trees vote, smoothing out individual tree errors.

### 4. Member 4 - Gradient Boosting Classifier
* **Core Concept**: A sequential boosting ensemble that builds decision trees sequentially, with each new shallow tree fitted to the negative gradient (pseudo-residuals) of the loss function.
* **Characteristics**: Reduces bias and variance in tandem; scales each tree's step by a learning rate $\eta$; produces state-of-the-art tabular accuracy.
* **Undergraduate Summary**: A team of learners where each new member specifically focuses on fixing the mistakes made by all previous members.


## 4. Baseline Results Comparison

Before hyperparameter optimization, each algorithm was trained with its default configuration to establish a benchmark floor.

### Default Configurations:
- **Logistic Regression**: `solver='lbfgs'`, `C=1.0`, `penalty='l2'`, `max_iter=1000`, `class_weight=None`
- **Decision Tree**: `criterion='gini'`, `max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`, `class_weight=None`
- **Random Forest**: `n_estimators=100`, `max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`, `class_weight=None`
- **Gradient Boosting**: `n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, `subsample=1.0`


In [ ]:
# Verified Baseline Results Table
baseline_df = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Accuracy': 0.8158,
        'Precision': 0.7060,
        'Recall': 0.5666,
        'F1-score': 0.6287,
        'ROC-AUC': 0.8776,
        'Train Acc': 0.8170,
        'Test Acc': 0.8158,
        'Overfitting Gap': 0.0012,
        'Generalization Note': 'Excellent generalization (gap 0.12 pp); lower recall'
    },
    {
        'Model': 'Decision Tree',
        'Accuracy': 0.8073,
        'Precision': 0.6480,
        'Recall': 0.6570,
        'F1-score': 0.6525,
        'ROC-AUC': 0.7625,
        'Train Acc': 0.9975,
        'Test Acc': 0.8073,
        'Overfitting Gap': 0.1901,
        'Generalization Note': 'Severe overfitting (gap 19.01 pp); memorized training data'
    },
    {
        'Model': 'Random Forest',
        'Accuracy': 0.8501,
        'Precision': 0.7826,
        'Recall': 0.6304,
        'F1-score': 0.6983,
        'ROC-AUC': 0.9059,
        'Train Acc': 0.9975,
        'Test Acc': 0.8501,
        'Overfitting Gap': 0.1474,
        'Generalization Note': 'Strong test performance, but severe overfitting (gap 14.74 pp)'
    },
    {
        'Model': 'Gradient Boosting',
        'Accuracy': 0.8394,
        'Precision': 0.7696,
        'Recall': 0.5948,
        'F1-score': 0.6710,
        'ROC-AUC': 0.8988,
        'Train Acc': 0.8381,
        'Test Acc': 0.8394,
        'Overfitting Gap': -0.0013,
        'Generalization Note': 'Well-regularized weak learners; virtually zero overfitting gap'
    }
])

print("Table 1: Verified Baseline Model Evaluation Results on Untouched Holdout Test Set")
baseline_df


## 5. Hyperparameter Optimization Summary

All four group members executed hyperparameter searches using **5-fold Stratified Cross-Validation on the training data exclusively ($X_{\text{train}}, y_{\text{train}}$)**. The test set was held strictly blind until model architectures were finalized.

### Optimization Strategy Overview
* **Scoring Metric**: All models used **$F_1$-score** as the optimization criterion. Because the dataset has a $72.5\% : 27.5\%$ majority-class imbalance, optimizing raw accuracy leads models to trivial majority prediction. $F_1$-score forces the search to balance precision and recall on the cancellation class.
* **Search Paradigms**: Exhaustive `GridSearchCV` was deployed for models with smaller discrete parameter spaces (Logistic Regression, Decision Tree). `RandomizedSearchCV` was deployed for complex ensembles (Random Forest, Gradient Boosting) to sample high-dimensional distributions efficiently within bounded compute budgets.


In [ ]:
# Hyperparameter Tuning Summary Table
tuning_summary_df = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Search Method': 'GridSearchCV (30 valid combos)',
        'CV Strategy': 'Stratified 5-Fold (shuffle=True, seed=42)',
        'Scoring Metric': 'F1-score',
        'Best CV F1': 0.6719,
        'Selected Hyperparameters': "C=1, penalty='l2', solver='liblinear', class_weight='balanced'"
    },
    {
        'Model': 'Decision Tree',
        'Search Method': 'GridSearchCV (252 combos / 1,260 fits)',
        'CV Strategy': 'Stratified 5-Fold (shuffle=True, seed=42)',
        'Scoring Metric': 'F1-score',
        'Best CV F1': 0.6994,
        'Selected Hyperparameters': "criterion='gini', max_depth=15, min_samples_split=2, min_samples_leaf=1, class_weight='balanced'"
    },
    {
        'Model': 'Random Forest',
        'Search Method': 'RandomizedSearchCV (15 iter / 75 fits)',
        'CV Strategy': 'Stratified 5-Fold (shuffle=True, seed=42)',
        'Scoring Metric': 'F1-score',
        'Best CV F1': 0.6822,
        'Selected Hyperparameters': "n_estimators=150, max_depth=20, min_samples_split=5, min_samples_leaf=4, max_features='sqrt', class_weight='balanced', bootstrap=True"
    },
    {
        'Model': 'Gradient Boosting',
        'Search Method': 'RandomizedSearchCV (12 iter / 60 fits)',
        'CV Strategy': 'Stratified 5-Fold (shuffle=True, seed=42)',
        'Scoring Metric': 'F1-score',
        'Best CV F1': 0.7076,
        'Selected Hyperparameters': "n_estimators=200, learning_rate=0.15, max_depth=4, min_samples_split=10, min_samples_leaf=4, subsample=0.9, max_features=None"
    }
])

print("Table 2: Systematic Hyperparameter Tuning Configuration and Cross-Validation Scores")
tuning_summary_df


## 6. Tuned Model Comparison

After selecting the optimal hyperparameters via 5-fold cross-validation, the final tuned estimators were evaluated on the **untouched holdout test set** ($17,446$ bookings).


In [ ]:
# Verified Tuned Model Results Table
tuned_df = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Accuracy': 0.7863,
        'Precision': 0.5772,
        'Recall': 0.8357,
        'F1-score': 0.6828,
        'ROC-AUC': 0.8776,
        'Train Acc': 0.7832,
        'Test Acc': 0.7863,
        'Overfitting Gap': -0.0031,
        'Best CV F1': 0.6719
    },
    {
        'Model': 'Decision Tree',
        'Accuracy': 0.7974,
        'Precision': 0.5895,
        'Recall': 0.8696,
        'F1-score': 0.7027,
        'ROC-AUC': 0.8783,
        'Train Acc': 0.8249,
        'Test Acc': 0.7974,
        'Overfitting Gap': 0.0275,
        'Best CV F1': 0.6994
    },
    {
        'Model': 'Random Forest',
        'Accuracy': 0.7835,
        'Precision': 0.5716,
        'Recall': 0.8521,
        'F1-score': 0.6842,
        'ROC-AUC': 0.8909,
        'Train Acc': 0.7921,
        'Test Acc': 0.7835,
        'Overfitting Gap': 0.0086,
        'Best CV F1': 0.6822
    },
    {
        'Model': 'Gradient Boosting',
        'Accuracy': 0.8501,
        'Precision': 0.7565,
        'Recall': 0.6714,
        'F1-score': 0.7114,
        'ROC-AUC': 0.9152,
        'Train Acc': 0.8562,
        'Test Acc': 0.8501,
        'Overfitting Gap': 0.0062,
        'Best CV F1': 0.7076
    }
])

print("Table 3: Verified Tuned Model Results on Untouched Holdout Test Set")
tuned_df


## 7. Model Comparison Visualizations

To provide visual clarity for presentation and defense, we plot four key comparative visualizations:
1. **Tuned Model Multi-Metric Comparison**: All 5 primary metrics across all 4 tuned models.
2. **Baseline vs. Tuned $F_1$-Score Comparison**: Demonstrating the impact of optimization on minority-class detection balance.
3. **Baseline vs. Tuned ROC-AUC Comparison**: Assessing discrimination capacity across operational thresholds.
4. **Generalization & Overfitting Gap Comparison**: Visualizing the reduction in training/test discrepancy achieved by tuning.


In [ ]:
# Visualization 1: Tuned Model Multi-Metric Comparison Bar Chart
fig, ax = plt.subplots(figsize=(11, 5.5))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']
models = tuned_df['Model'].tolist()
x = np.arange(len(models))
width = 0.16

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

for i, metric in enumerate(metrics):
    offset = (i - 2) * width
    rects = ax.bar(x + offset, tuned_df[metric], width, label=metric, color=colors[i], edgecolor='black', linewidth=0.6)
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, rotation=0)

ax.set_ylabel('Score (0.00 - 1.00)', fontweight='bold')
ax.set_title('Figure 1: Tuned Model Multi-Metric Performance on Holdout Test Set', pad=14)
ax.set_xticks(x)
ax.set_xticklabels(models, fontweight='bold')
ax.set_ylim(0.40, 1.02)
ax.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.9)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# Visualizations 2, 3, and 4: Paired Comparisons
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

x = np.arange(len(models))
width = 0.35

# Chart 2: Baseline vs Tuned F1-score
b_f1 = baseline_df['F1-score'].values
t_f1 = tuned_df['F1-score'].values

axes[0].bar(x - width/2, b_f1, width, label='Baseline', color='#A1BAE3', edgecolor='black', linewidth=0.6)
axes[0].bar(x + width/2, t_f1, width, label='Tuned', color='#2B5C8F', edgecolor='black', linewidth=0.6)
axes[0].set_title('Figure 2: Baseline vs. Tuned F1-Score')
axes[0].set_ylabel('F1-Score')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=20, ha='right')
axes[0].set_ylim(0.50, 0.80)
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
for i in range(len(models)):
    axes[0].annotate(f"{b_f1[i]:.3f}", (x[i]-width/2, b_f1[i]), xytext=(0,2), textcoords="offset points", ha='center', fontsize=8)
    axes[0].annotate(f"{t_f1[i]:.3f}", (x[i]+width/2, t_f1[i]), xytext=(0,2), textcoords="offset points", ha='center', fontsize=8, fontweight='bold')

# Chart 3: Baseline vs Tuned ROC-AUC
b_auc = baseline_df['ROC-AUC'].values
t_auc = tuned_df['ROC-AUC'].values

axes[1].bar(x - width/2, b_auc, width, label='Baseline', color='#A9DFBF', edgecolor='black', linewidth=0.6)
axes[1].bar(x + width/2, t_auc, width, label='Tuned', color='#1E8449', edgecolor='black', linewidth=0.6)
axes[1].set_title('Figure 3: Baseline vs. Tuned ROC-AUC')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=20, ha='right')
axes[1].set_ylim(0.70, 0.98)
axes[1].legend(loc='lower right')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
for i in range(len(models)):
    axes[1].annotate(f"{b_auc[i]:.3f}", (x[i]-width/2, b_auc[i]), xytext=(0,2), textcoords="offset points", ha='center', fontsize=8)
    axes[1].annotate(f"{t_auc[i]:.3f}", (x[i]+width/2, t_auc[i]), xytext=(0,2), textcoords="offset points", ha='center', fontsize=8, fontweight='bold')

# Chart 4: Overfitting Gap (Train Acc - Test Acc)
b_gap = baseline_df['Overfitting Gap'].values * 100
t_gap = tuned_df['Overfitting Gap'].values * 100

axes[2].bar(x - width/2, b_gap, width, label='Baseline Gap', color='#F5B7B1', edgecolor='black', linewidth=0.6)
axes[2].bar(x + width/2, t_gap, width, label='Tuned Gap', color='#B03A2E', edgecolor='black', linewidth=0.6)
axes[2].set_title('Figure 4: Overfitting Gap Reduction (pp)')
axes[2].set_ylabel('Train Acc - Test Acc (percentage points)')
axes[2].set_xticks(x)
axes[2].set_xticklabels(models, rotation=20, ha='right')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].legend(loc='upper right')
axes[2].grid(axis='y', linestyle='--', alpha=0.7)
for i in range(len(models)):
    axes[2].annotate(f"{b_gap[i]:+.1f}", (x[i]-width/2, max(b_gap[i], 0)), xytext=(0,2), textcoords="offset points", ha='center', fontsize=8)
    axes[2].annotate(f"{t_gap[i]:+.1f}", (x[i]+width/2, max(t_gap[i], 0)), xytext=(0,2), textcoords="offset points", ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Confusion Matrix Analysis

In hotel revenue management, not all classification errors have equal business consequences:
* **True Negative (TN)**: Model predicts booking will stay; guest arrives. (Room occupied, revenue earned).
* **False Positive (FP - Type I Error)**: Model predicts cancellation; guest arrives unexpectedly. (Risk of customer dissatisfaction if room is resold or staff unprepared).
* **False Negative (FN - Type II Error)**: Model predicts guest arrives; guest cancels unexpectedly. (Room sits vacant, perishable room night is permanently lost without revenue).
* **True Positive (TP)**: Model detects cancellation in advance; booking cancels. (Hotel can resell the room or implement mitigation strategies).

### Tuned Confusion Matrices on Holdout Test Set ($N=17,446$):

| Metric / Quadrant | Logistic Regression | Decision Tree | Random Forest | Gradient Boosting |
| :--- | :---: | :---: | :---: | :---: |
| **True Negatives (TN)** | $9,705$ | $9,736$ | $9,577$ | **$11,606$** |
| **False Positives (FP)** | $2,939$ | $2,908$ | $3,067$ | **$1,038$** |
| **False Negatives (FN)** | $789$ | **$626$** | $710$ | $1,578$ |
| **True Positives (TP)** | $4,013$ | **$4,176$** | $4,092$ | $3,224$ |
| **Cancellations Caught (Recall)** | $83.57\%$ | **$86.96\%$** | $85.21\%$ | $67.14\%$ |
| **False Alarms Raised (FP)** | $2,939$ | $2,908$ | $3,067$ | **$1,038$** |


In [ ]:
# Visualizing Confusion Matrices side-by-side using pure matplotlib
cm_dict = {
    'Logistic Regression': np.array([[9705, 2939], [789, 4013]]),
    'Decision Tree': np.array([[9736, 2908], [626, 4176]]),
    'Random Forest': np.array([[9577, 3067], [710, 4092]]),
    'Gradient Boosting': np.array([[11606, 1038], [1578, 3224]])
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))

for i, (name, cm) in enumerate(cm_dict.items()):
    ax = axes[i]
    cmap = plt.cm.Greens if i == 3 else plt.cm.Blues
    im = ax.imshow(cm, cmap=cmap, aspect='auto')
    
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    
    # Text annotations in each cell
    labels = [
        [f"TN: {tn:,}\n(Correct Stay)", f"FP: {fp:,}\n(False Alarm)"],
        [f"FN: {fn:,}\n(Missed Cancel)", f"TP: {tp:,}\n(Caught Cancel)"]
    ]
    
    thresh = cm.max() / 2.0
    for r in range(2):
        for c in range(2):
            color = "white" if cm[r, c] > thresh else "black"
            ax.text(c, r, labels[r][c], ha="center", va="center", color=color, fontsize=9, fontweight='bold')
            
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0 (Stay)', 'Pred 1 (Cancel)'], fontsize=9)
    ax.set_yticklabels(['True 0 (Stay)', 'True 1 (Cancel)'], fontsize=9)
    ax.set_title(f"{name}\n(Acc: {tuned_df.loc[i, 'Accuracy']:.4f} | Rec: {tuned_df.loc[i, 'Recall']:.4f})", pad=8)
    ax.grid(False)

plt.suptitle('Figure 5: Tuned Model Confusion Matrices on Holdout Test Set (N=17,446)', fontsize=14, y=1.04, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Baseline vs. Tuned Model Analysis

Each model's response to hyperparameter tuning illustrates a distinct data mining dynamic:

### 1. Logistic Regression
* **What changed**: Incorporating `class_weight='balanced'` shifted the decision threshold dynamically to counter class imbalance.
* **Metric Shift**: Accuracy dropped slightly ($81.58\% \rightarrow 78.63\%$) and Precision decreased ($70.60\% \rightarrow 57.72\%$), but Recall surged dramatically from **$56.66\%$ to $83.57\%$**, elevating the balanced $F_1$-score from $0.6287$ to **$0.6828$**. ROC-AUC remained identical ($0.8776$).
* **Undergraduate Takeaway**: A classic precision-recall trade-off. The model willingly tolerates more false alarms to avoid missing cancellations.

### 2. Decision Tree
* **What changed**: Constraining maximum depth (`max_depth=15`) and applying `class_weight='balanced'` replaced an unconstrained tree that had grown until leaves were pure.
* **Metric Shift**: Training accuracy dropped from $99.75\%$ to $82.49\%$, collapsing the overfitting gap from **$+19.01$ percentage points down to $+2.75$ pp**. Test $F_1$ increased from $0.6525$ to **$0.7027$**, Recall surged to **$86.96\%$**, and ROC-AUC jumped from $0.7625$ to **$0.8783$**.
* **Undergraduate Takeaway**: Pruning successfully broke memorization, producing a robust tree that generalizes well to new data.

### 3. Random Forest
* **What changed**: Adding depth limits (`max_depth=20`), leaf constraints (`min_samples_leaf=4`), and `class_weight='balanced'` forced individual trees to remain shallower and prioritize minority samples.
* **Metric Shift**: Training accuracy dropped from $99.75\%$ to $79.21\%$, eliminating the massive baseline overfitting gap (**$+14.74$ pp reduced to $+0.86$ pp**). Recall jumped from $63.04\%$ to **$85.21\%$**. However, Accuracy dropped from $85.01\%$ to $78.35\%$, Precision dropped from $78.26\%$ to $57.16\%$, and ROC-AUC dropped slightly from $0.9059$ to $0.8909$.
* **Undergraduate Takeaway**: **Tuning Random Forest created an explicit trade-off, not a simple Pareto improvement.** Tuning prioritized high cancellation detection at the expense of overall precision. It should not be labeled 'strictly better' without contextual qualification.

### 4. Gradient Boosting
* **What changed**: Regularizing tree depth (`max_depth=4`), introducing stochastic subsampling (`subsample=0.9`), and tuning learning rate (`learning_rate=0.15`) with 200 trees.
* **Metric Shift**: Accuracy rose from $83.94\%$ to **$85.01\%$**, Recall rose from $59.48\%$ to **$67.14\%$**, $F_1$-score rose from $0.6710$ to **$0.7114$**, and ROC-AUC rose to **$0.9152$** (highest in the project). The overfitting gap remained virtually zero (**$+0.62$ pp**).
* **Undergraduate Takeaway**: Gradient Boosting achieved unambiguous improvements across nearly all dimensions while maintaining tight train/test alignment.


## 10. Overfitting Analysis

Overfitting occurs when an estimator memorizes noise and idiosyncratic variations in the training sample rather than underlying structural patterns, leading to degradation on unseen data.

$$\text{Overfitting Gap} = \text{Train Accuracy} - \text{Test Accuracy}$$

### Quantitative Overfitting Summary:
1. **Decision Tree Baseline**:
   * Train Acc: $99.75\%$, Test Acc: $80.73\%$, **Gap: $+19.01$ percentage points**.
   * An unrestricted tree split until leaves contained isolated samples, capturing pure noise. Tuning to depth 15 slashed this gap by **$85.5\%$** (to $+2.75$ pp).
2. **Random Forest Baseline**:
   * Train Acc: $99.75\%$, Test Acc: $85.01\%$, **Gap: $+14.74$ percentage points**.
   * While bagging reduced variance compared to a single tree, unconstrained trees still memorized training patterns. Tuning constrained leaves and tree depth, shrinking the gap by **$94.2\%$** (to $+0.86$ pp).
3. **Logistic Regression & Gradient Boosting**:
   * Both algorithms showed exceptional generalization from the start.
   * Baseline gaps were $+0.12$ pp and $-0.13$ pp respectively.
   * *Note on small negative gaps*: A test accuracy fractionally higher than training accuracy (e.g. $-0.13$ to $-0.31$ pp) is standard statistical variance on large samples, not an error.


## 11. Metric Explanation: Why Accuracy Alone is Insufficient

In binary classification with imbalanced outcomes, evaluating models solely by Accuracy is misleading:

* **Accuracy (Overall Correctness)**:
  $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
  * *Limitation*: A naive classifier predicting 'Never Cancel' on this dataset achieves **$72.48\%$ accuracy while catching 0 cancellations**.
* **Precision (Reliability of Cancellation Alarms)**:
  $$\text{Precision} = \frac{TP}{TP + FP}$$
  * Answers: *When the model flags a reservation as cancelled, how often is it right?* High precision avoids alienating guests with unnecessary pre-payments or overbooking rooms when guests actually show up.
* **Recall / Sensitivity (Catch Rate of Cancellations)**:
  $$\text{Recall} = \frac{TP}{TP + FN}$$
  * Answers: *Of all bookings that truly cancelled, what fraction did the model catch?* Crucial for inventory management so the hotel can release and resell perishable room nights.
* **$F_1$-Score (Harmonic Mean)**:
  $$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
  * Balances both error types into a single metric without being fooled by majority class skew.
* **ROC-AUC (Discrimination Across Thresholds)**:
  * Measures the probability that a randomly chosen cancelled booking receives a higher predicted cancellation score than a randomly chosen non-cancelled booking across all potential decision thresholds.


## 12. Final Model Selection Analysis

To select the proposed final model for hotel deployment, we systematically evaluate the complete multi-dimensional evidence base:

### Comparative Trade-Off Matrix:
1. **Decision Tree (Tuned)**:
   * **Strengths**: Highest cancellation catch rate (Recall = **$86.96\%$**; misses only 626 cancellations). Transparent visual rule paths.
   * **Weaknesses**: Lowest Precision ($58.95\%$); generates **2,908 false alarms**. Modest overall accuracy ($79.74\%$).
2. **Random Forest (Tuned)**:
   * **Strengths**: High Recall ($85.21\%$); excellent ROC-AUC ($0.8909$); negligible overfitting gap ($0.86$ pp).
   * **Weaknesses**: Lowest accuracy ($78.35\%$) and precision ($57.16\%$); generates the highest false alarms (**3,067**). Suffered trade-offs during tuning.
3. **Logistic Regression (Tuned)**:
   * **Strengths**: Extremely fast inference; linear coefficients provide direct interpretability; good recall ($83.57\%$).
   * **Weaknesses**: Lowest cross-validation score ($0.6719$); lower precision ($57.72\%$).
4. **Gradient Boosting (Tuned)**:
   * **Strengths**:
     * **Highest Test Accuracy ($85.01\%$)**
     * **Highest $F_1$-Score ($0.7114$)**
     * **Highest ROC-AUC ($0.9152$)**
     * **Highest Precision ($75.65\%$)**
     * **Highest 5-Fold CV Score ($0.7076$)**
     * **Minimal Overfitting Gap ($+0.0062$)**
     * **Lowest False Alarms**: Only 1,038 false alarms (vs. $\sim 3,000$ for all other models).
   * **Trade-Off**: Moderate Recall ($67.14\%$) catches 3,224 cancellations, leaving 1,578 missed cancellations at the default 0.50 threshold.

### Strategic Revenue Management Justification:
In practical hotel operations, an algorithm that generates 3,000 false alarms on 17,000 bookings creates operational chaos: front desk staff overbook aggressively, risking walk-in rejections and customer backlash.

**Tuned Gradient Boosting is selected as the Proposed Final Model** because:
1. It is the top-performing model across **Accuracy, Precision, $F_1$-score, ROC-AUC, and Cross-Validation**.
2. It generates almost **one-third the false alarms** of any competitor, ensuring reliable operational trust.
3. Because its **ROC-AUC is an exceptional 0.9152**, hotel management can adjust the operational probability threshold downward (e.g. from 0.50 to 0.35) during high-demand peak seasons to capture more cancellations without altering the underlying model.


## 13. Final Comparison Table

The report-ready summary of all tuned models evaluated on the common holdout test set ($N=17,446$):


In [ ]:
# Presentation-ready summary table
final_summary_table = pd.DataFrame([
    {
        'Rank': '1 (Selected)',
        'Model': 'Gradient Boosting',
        'Accuracy': 0.8501,
        'Precision': 0.7565,
        'Recall': 0.6714,
        'F1-score': 0.7114,
        'ROC-AUC': 0.9152,
        'Train Acc': 0.8562,
        'Overfitting Gap': 0.0062,
        'Best CV F1': 0.7076,
        'Operational Role': 'Optimal balanced production model'
    },
    {
        'Rank': '2',
        'Model': 'Decision Tree',
        'Accuracy': 0.7974,
        'Precision': 0.5895,
        'Recall': 0.8696,
        'F1-score': 0.7027,
        'ROC-AUC': 0.8783,
        'Train Acc': 0.8249,
        'Overfitting Gap': 0.0275,
        'Best CV F1': 0.6994,
        'Operational Role': 'Maximum recall / rule-based auditing'
    },
    {
        'Rank': '3',
        'Model': 'Random Forest',
        'Accuracy': 0.7835,
        'Precision': 0.5716,
        'Recall': 0.8521,
        'F1-score': 0.6842,
        'ROC-AUC': 0.8909,
        'Train Acc': 0.7921,
        'Overfitting Gap': 0.0086,
        'Best CV F1': 0.6822,
        'Operational Role': 'High recall ensemble alternative'
    },
    {
        'Rank': '4',
        'Model': 'Logistic Regression',
        'Accuracy': 0.7863,
        'Precision': 0.5772,
        'Recall': 0.8357,
        'F1-score': 0.6828,
        'ROC-AUC': 0.8776,
        'Train Acc': 0.7832,
        'Overfitting Gap': -0.0031,
        'Best CV F1': 0.6719,
        'Operational Role': 'Linear transparent benchmark'
    }
]).set_index('Rank')

print("Table 4: Comprehensive Model Ranking and Evaluation 2 Final Selection Matrix")
final_summary_table


## 14. Member 3 - Random Forest Viva Focus

This section is prepared specifically for **Member 3 (Random Forest)** to provide thorough, defensible answers during the Evaluation 2 oral examination.

### 1. What is Random Forest?
Random Forest is a bagging (Bootstrap Aggregating) ensemble of decision trees. It creates diversity among trees in two ways:
1. **Bootstrapping**: Each tree is trained on a random bootstrap sample (with replacement) of the training records.
2. **Feature Subsampling (`max_features='sqrt'`)**: At every single node split, only a random subset of features is evaluated. This decorrelates the trees so that strong dominant predictors (e.g. `deposit_type_Non_Refund`) do not dominate every tree in the forest.

### 2. Baseline vs. Tuned Results Comparison
* **Baseline RF**: Accuracy = **$0.8501$**, Precision = **$0.7826$**, Recall = **$0.6304$**, $F_1$ = **$0.6983$**, ROC-AUC = **$0.9059$**, Train Acc = **$0.9975$**, Gap = **$+0.1474$**.
* **Tuned RF**: Accuracy = **$0.7835$**, Precision = **$0.5716$**, Recall = **$0.8521$**, $F_1$ = **$0.6842$**, ROC-AUC = **$0.8909$**, Train Acc = **$0.7921$**, Gap = **$+0.0086$**.

### 3. Key Viva Explanations for Member 3:
* **Why Baseline Training Accuracy was $99.75\%$**:
  Because trees were unconstrained (`max_depth=None`, `min_samples_split=2`), individual trees split until virtually every leaf was pure, memorizing training records.
* **Why the $+14.74$ pp Gap Proves Overfitting**:
  A nearly $15$ percentage-point spread between training accuracy ($99.75\%$) and test accuracy ($85.01\%$) indicates high variance. The ensemble was fitting to idiosyncratic noise.
* **How Tuning Slashed the Gap from $0.1474$ to $0.0086$**:
  By restricting tree depth to 20 (`max_depth=20`) and enforcing at least 4 samples per leaf (`min_samples_leaf=4`), leaves could no longer isolate individual outliers, forcing trees to learn generalizable decision boundaries.
* **Why Recall Increased from $0.6304$ to $0.8521$**:
  The hyperparameter search optimized for $F_1$-score with `class_weight='balanced'`. This penalized false negatives more heavily, shifting the forest voting distribution to catch $4,092$ cancellations (up from $3,027$).
* **Why Accuracy and Precision Decreased**:
  Prioritizing minority-class cancellation detection caused the model to flag more marginal reservations as cancellations, increasing false positives from $841$ to $3,067$, which lowered Precision ($78.26\% \rightarrow 57.16\%$) and overall Accuracy ($85.01\% \rightarrow 78.35\%$).
* **Why Tuned RF is NOT Automatically 'Better' Than Baseline**:
  This represents an **algorithmic trade-off**, not a pure improvement. If a hotel prioritizes overall accuracy and low false alarms, the baseline configuration is preferable. If the hotel prioritizes catching every possible cancellation, the tuned configuration is superior.
* **Random Forest vs. Decision Tree**:
  A Decision Tree is a single high-variance learner prone to overfitting. Random Forest combines 150 decorrelated trees, averaging their predictions to reduce variance.
* **Random Forest vs. Gradient Boosting**:
  Random Forest trains trees **in parallel and independently** using bagging. Gradient Boosting trains trees **sequentially**, with each tree fitted to the residual errors of prior trees using boosting.
* **What the Selected Hyperparameters Do**:
  * `n_estimators=150`: Uses 150 trees to ensure stable variance reduction without excessive compute time.
  * `max_depth=20`: Caps the hierarchy of questions, preventing runaway branch memorization.
  * `min_samples_leaf=4`: Guarantees terminal nodes represent at least 4 bookings, smoothing boundaries.
  * `max_features='sqrt'`: Samples $\approx \sqrt{899} \approx 30$ features per split, ensuring tree independence.
  * `class_weight='balanced'`: Counteracts the $72.5 / 27.5$ class imbalance.


## 15. Evaluation 2 Viva Preparation: Core Questions & Answers

Concise, plain-English answers for undergraduate oral defense:

#### Q1: Why did we implement four algorithms instead of just one?
* **A**: To benchmark across diverse machine learning paradigms: linear modeling (Logistic Regression), single tree partitioning (Decision Tree), bagging ensemble (Random Forest), and sequential boosting (Gradient Boosting). Different algorithms make different inductive assumptions about data structure.

#### Q2: Why must all four models use the exact same test set?
* **A**: For fair and scientifically valid comparison. Evaluating on different splits introduces variance in sample difficulty, class ratios, and noise, confounding model comparison.

#### Q3: What is a baseline model?
* **A**: An initial model trained with default hyperparameter configurations. It serves as a performance floor to evaluate whether hyperparameter tuning actually delivers real benefits.

#### Q4: What is hyperparameter tuning?
* **A**: The systematic search for optimal configuration values (like tree depth or regularisation strength) that govern model architecture and learning, which are not learned during standard training.

#### Q5: What is cross-validation, and why use Stratified K-Fold?
* **A**: Cross-validation divides the training set into $K$ folds to validate models iteratively without touching the test set. Stratified K-Fold ensures each fold preserves the empirical class ratio ($27.5\%$ cancellations), preventing fold bias on imbalanced data.

#### Q6: Why must the test set remain untouched during tuning?
* **A**: Using test data to pick hyperparameters leaks test information into model selection, producing dishonestly optimistic performance estimates that fail in production.

#### Q7: What is overfitting, and which baseline models suffered from it?
* **A**: Overfitting is when a model memorizes training noise rather than generalizable patterns. Baseline Decision Tree (gap $+19.01$ pp) and Baseline Random Forest (gap $+14.74$ pp) both suffered severe overfitting due to unconstrained tree depth.

#### Q8: Why is Accuracy insufficient for this problem?
* **A**: Because the dataset is imbalanced ($72.5\%$ non-cancellations). A naive model predicting 'Never Cancel' gets $72.5\%$ accuracy while catching zero cancellations, rendering it useless for hotel revenue management.

#### Q9: What is the difference between Precision and Recall for hotel cancellations?
* **A**: **Precision** measures alarm accuracy (when we predict cancellation, did they truly cancel?). **Recall** measures catch rate (out of all true cancellations, how many did we identify?).

#### Q10: What are False Positives and False Negatives in this business context?
* **A**: A **False Positive** is a false alarm (guest is predicted to cancel but arrives to check in). A **False Negative** is a missed cancellation (guest cancels unexpectedly, leaving a vacant room that could have been resold).

#### Q11: What is the fundamental difference between Random Forest and Gradient Boosting?
* **A**: Random Forest trains trees **in parallel** via bagging and averages them to reduce variance. Gradient Boosting trains trees **sequentially** via boosting, with each tree correcting the residual errors of preceding trees to reduce both bias and variance.

#### Q12: Which final model was selected and why?
* **A**: **Tuned Gradient Boosting**. It achieved the highest overall Accuracy ($85.01\%$), highest $F_1$-score ($0.7114$), highest ROC-AUC ($0.9152$), highest Precision ($75.65\%$), and best CV score ($0.7076$), while maintaining a negligible overfitting gap ($0.62$ pp) and generating two-thirds fewer false alarms than competing models.


In [ ]:
# Export and verify the results CSV file
csv_export_path = REPO_ROOT / 'results/modeling/model_comparison.csv'

# Combine baseline and tuned records into comprehensive results table
export_df = pd.concat([
    baseline_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'Train Acc', 'Test Acc', 'Overfitting Gap']].assign(
        stage='Baseline', cv_f1=np.nan
    ),
    tuned_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'Train Acc', 'Test Acc', 'Overfitting Gap', 'Best CV F1']].rename(
        columns={'Best CV F1': 'cv_f1'}
    ).assign(stage='Tuned')
], ignore_index=True)

# Align column names to exact project standard
export_df.rename(columns={
    'Model': 'model',
    'Accuracy': 'accuracy',
    'Precision': 'precision',
    'Recall': 'recall',
    'F1-score': 'f1',
    'ROC-AUC': 'roc_auc',
    'Train Acc': 'train_accuracy',
    'Test Acc': 'test_accuracy',
    'Overfitting Gap': 'overfitting_gap'
}, inplace=True)

# Reorder columns
col_order = ['model', 'stage', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'train_accuracy', 'test_accuracy', 'overfitting_gap', 'cv_f1']
export_df = export_df[col_order]

# Ensure directory exists and save
csv_export_path.parent.mkdir(parents=True, exist_ok=True)
export_df.to_csv(csv_export_path, index=False)

print(f"Verified model comparison results successfully written to: {csv_export_path}")
print(f"Total exported rows: {len(export_df)}")
print(pd.read_csv(csv_export_path).to_string())
